# Transpilación de Random Forest a C++

Este cuadernillo carga el modelo Random Forest entrenado (`rf_model.bin`) y lo transpila a código C++ utilizando `micromlgen` para su integración directa en el ESP32-S3.

## 1. Carga de Librerías y Variables de Entorno

In [1]:
import os
import joblib
from micromlgen import port

def load_env_variables():
    from pathlib import Path
    try:
        start_dir = Path(os.getcwd())
    except:
        start_dir = Path(".")
        
    env_path = None
    for path in [start_dir] + list(start_dir.parents):
        temp_path = path / ".env"
        if temp_path.exists():
            env_path = temp_path
            break
            
    if env_path is None:
        raise FileNotFoundError("⚠️ No se pudo encontrar el archivo .env en la raíz del proyecto.")
        
    with open(env_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            if "=" not in line:
                continue
            key, val = line.split("=", 1)
            os.environ[key.strip()] = val.strip()
            
    print(f"✅ Archivo .env cargado con éxito desde: {env_path}")

load_env_variables()


✅ Archivo .env cargado con éxito desde: /home/cbe/Proyectos/MyoTensor_Tesis/.env


## 2. Carga del Modelo Random Forest Exportado

In [2]:
models_dir = os.path.join(os.environ["PROJECT_ROOT"], "intelligence", "models", "myotensor_proto", "ml")
model_path = os.path.join(models_dir, "rf_model.bin")

print(f"📖 Cargando el modelo RF desde: {model_path} ...")
rf_model = joblib.load(model_path)
print("✅ Modelo cargado exitosamente:")
print(rf_model)


📖 Cargando el modelo RF desde: /home/cbe/Proyectos/MyoTensor_Tesis/intelligence/models/myotensor_proto/ml/rf_model.bin ...
✅ Modelo cargado exitosamente:
RandomForestClassifier(class_weight='balanced', max_depth=9, min_samples_leaf=2,
                       n_estimators=40, random_state=42)


## 3. Transpilación a C++ con micromlgen

In [3]:
print("⚙️ Transpilando el modelo a código C++ ...")
cpp_code = port(rf_model)
print("✅ Transpilación completada. Vista previa de las primeras 500 líneas:\n")
print("\n".join(cpp_code.split("\n")[:500]))


⚙️ Transpilando el modelo a código C++ ...
✅ Transpilación completada. Vista previa de las primeras 500 líneas:

#pragma once
#include <cstdarg>
namespace Eloquent {
    namespace ML {
        namespace Port {
            class RandomForest {
                public:
                    /**
                    * Predict class for features vector
                    */
                    int predict(float *x) {
                        uint8_t votes[4] = { 0 };
                        // tree #1
                        if (x[1] <= 0.2518711984157562) {
                            if (x[0] <= 0.1516656056046486) {
                                if (x[0] <= 0.1366715207695961) {
                                    votes[0] += 1;
                                }

                                else {
                                    if (x[0] <= 0.13674497604370117) {
                                        votes[2] += 1;
                                    }

                         

## 4. Guardar archivo de encabezado (.h) en el Firmware

In [4]:
firmware_include_dir = os.path.join(os.environ["PROJECT_ROOT"], "firmware", "Classifier", "include")
os.makedirs(firmware_include_dir, exist_ok=True)

header_path = os.path.join(firmware_include_dir, "rf_model.h")

print(f"💾 Escribiendo código C++ en: {header_path} ...")
with open(header_path, "w", encoding="utf-8") as f:
    f.write(cpp_code)
    
print("✅ Archivo header escrito exitosamente. Listo para compilar en PlatformIO.")


💾 Escribiendo código C++ en: /home/cbe/Proyectos/MyoTensor_Tesis/firmware/Classifier/include/rf_model.h ...
✅ Archivo header escrito exitosamente. Listo para compilar en PlatformIO.
